# 🔬 PUF Fotónica SmartLight — Medida SIN Anillo Isotrópico
**TFM · Oxel Arnal Martínez · ITEAM/PRL · UPV · 2025**

---

## Propósito de este notebook

Medir los **mismos 500 retos** que en el experimento del anillo isotrópico
pero **sin programar el routing del anillo** — solo se aplican las fases
dinámicas del reto en las TBUs dinámicas. Las TBUs de ruta quedan en
su estado de reset (solo φ₀ activa, sin corriente).

Esto permite comparar directamente:
- **Con anillo**: `powers_anillo_COMPLETO_*.json` (ya medido)
- **Sin anillo**: `powers_sin_anillo_*.json` (este experimento)

La única diferencia es la presencia/ausencia del routing del anillo isotrópico.

---

| Celda | Qué hace | ¿Cuántas veces? |
|-------|----------|--------------------|
| 0 | Imports y parámetros | 1 vez |
| 1 | Conectar chip | 1 vez |
| 2 | Verificar polarización (ajuste manual) | Cada puerto |
| 3 | Comprobar polarización con fotodetectores | Cada puerto |
| 4 | **MEDIR — bucle sin anillo** | Cada puerto |
| 5 | Estado de progreso | Cuando quieras |
| 6 | Desconectar | Al final |

> **Cuando cambies de puerto:** modifica `INPORT` en la **Celda 0**
> y ejecuta desde la **Celda 2**.


## Celda 0 — Imports y parámetros
Ejecutar **una sola vez** al abrir el cuaderno. Cambiar `INPORT` cada vez que cambies el puerto físico.

In [ ]:
import json, random, time, pathlib
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from smartlight import Smartlight

# ── Puerto de entrada activo ──────────────────────────────────────────
# ⚠️ CAMBIAR ESTE VALOR cada vez que cambies el setup físico del lab
INPORT = 3

# ── Parámetros fijos ──────────────────────────────────────────────────
ACTIVE_PORTS = list(range(0, 22)) + list(range(34, 40))  # 28 puertos
N_CHALLENGES = 500    # mismos 500 retos que en el experimento del anillo
SEED         = 44     # misma seed que en retos_v6 → mismos retos
FASE_MAX     = 2 * np.pi

# ── Rutas de archivos ─────────────────────────────────────────────────
RETOS_DIR   = pathlib.Path('retos_v6')          # challenges ya generados
RESULTS_DIR = pathlib.Path('resultados_sin_anillo')
RESULTS_DIR.mkdir(exist_ok=True)

# ── Puertos de salida para el INPORT actual ───────────────────────────
outports = [p for p in ACTIVE_PORTS if p != INPORT]

print(f'Puerto activo:      {INPORT}')
print(f'Puertos de salida:  {len(outports)}')
print(f'Retos a medir:      {N_CHALLENGES}')
print(f'Resultados en:      {RESULTS_DIR}/')

# ── Ver puertos ya medidos ────────────────────────────────────────────
medidos = sorted([
    int(f.stem.split('sin_anillo_port')[1].split('.')[0])
    for f in RESULTS_DIR.glob('powers_sin_anillo_port*.json')
])
pendientes = [p for p in ACTIVE_PORTS if p not in medidos]
print(f'\nPuertos ya medidos:  {medidos}')
print(f'Puertos pendientes:  {pendientes}')
if pendientes and INPORT not in medidos:
    print(f'\n→ INPORT = {INPORT} ← listo para medir')
elif INPORT in medidos:
    print(f'\n⚠️  Puerto {INPORT} ya medido — cambia INPORT si quieres medir otro')


## Celda 1 — Conectar el chip
Ejecutar **una sola vez** al inicio de la sesión.

In [ ]:
sl = Smartlight()
sl.connect()
sl.calibration()
sl.reset_mesh()
print(f'✓ Chip conectado y calibrado')
print(f'reset_mesh() — todas las TBUs en φ₀ puro (sin corriente)')


## Celda 2 — Ajuste de polarización
Ejecutar **cada vez que cambies INPORT**.

Configura el chip en modo externo para que puedas ajustar los paddles
mirando el medidor analógico.

In [ ]:
# Puerto de salida para verificar la polarización
OUTPORT_POL = outports[0]

sl.reset_mesh()
sl.set_input_port(INPORT)
sl.enable_external_monitoring([OUTPORT_POL])
sl.interconnect_auto(INPORT, OUTPORT_POL)

print(f'✓ Chip en modo externo')
print(f'  Puerto entrada: {INPORT}')
print(f'  Puerto salida test: {OUTPORT_POL}')
print(f'\n→ Ajusta los paddles mirando el medidor analógico')
print(f'→ Cuando la señal esté maximizada, ejecuta la Celda 3')


## Celda 3 — Comprobar polarización con fotodetectores internos
Ejecutar tras ajustar los paddles. Verificar que la señal es buena.

In [ ]:
sl.reset_mesh()
sl.set_input_port(INPORT)
sl.enable_internal_monitoring()
sl.interconnect_auto(INPORT, OUTPORT_POL)

powers_check = sl.get_output_power(inport=INPORT, outport=outports)

print(f'Potencias actuales — Puerto entrada: {INPORT}')
print(f'{"Puerto":>8}  {"Potencia (dBm)":>15}')
print('-' * 30)
vals = []
for p in sorted(outports):
    v   = powers_check.get(p, powers_check.get(str(p), -60.0))
    bar = '█' * max(0, int((v + 50) / 2))
    print(f'{p:>8}  {v:>15.4f}  {bar}')
    vals.append(v)

print(f'\nMin:   {min(vals):.2f} dBm  → puerto {outports[np.argmin(vals)]}')
print(f'Max:   {max(vals):.2f} dBm  → puerto {outports[np.argmax(vals)]}')
print(f'Media: {np.mean(vals):.2f} dBm')
print(f'\n→ ¿Polarización OK? Ejecuta la Celda 4 para medir')


## Celda 4 — MEDIR sin anillo isotrópico
**Modo sin anillo:** se aplican las mismas fases dinámicas que en el experimento
del anillo, pero las TBUs de ruta NO se programan — quedan en reset (solo φ₀).

Los retos se cargan del mismo fichero `retos_v6/challenges_ring_port{XX}.json`
generado con `SEED=44`, garantizando que los retos son exactamente los mismos
que en el experimento con el anillo.

> Tiempo estimado: ~500 × 0.5s ≈ 4-5 minutos por puerto.

In [ ]:
# ── VERIFICAR que no está ya medido ──────────────────────────────────
fout = RESULTS_DIR / f'powers_sin_anillo_port{INPORT:02d}.json'
if fout.exists():
    print(f'⚠️  Puerto {INPORT} ya medido en {fout.name}')
    print(f'   Cambia INPORT en Celda 0 para medir otro puerto')
    print(f'   O borra el fichero si quieres repetir la medida')
else:
    # ── CARGAR CHALLENGES (mismos que con el anillo) ──────────────────
    fname_chall = RETOS_DIR / f'challenges_ring_port{INPORT:02d}.json'
    if not fname_chall.exists():
        print(f'✗ No se encuentra {fname_chall}')
        print(f'  Verifica que la carpeta retos_v6/ existe y tiene los archivos')
    else:
        with open(fname_chall) as f:
            d_chall = json.load(f)

        challenges   = d_chall['challenges']
        dynamic_pucs = d_chall['dynamic_pucs']
        route_pucs   = d_chall['route_pucs']

        print(f'✓ Challenges cargados de {fname_chall.name}')
        print(f'  Total retos:      {len(challenges)}')
        print(f'  TBUs dinámicas:   {len(dynamic_pucs)}')
        print(f'  TBUs de ruta:     {len(route_pucs)}  ← NO se programan en este exp.')
        print(f'  Seed:             {d_chall["seed"]}')

        # ── VERIFICAR que hay suficientes retos ──────────────────────────
        if len(challenges) < N_CHALLENGES:
            print(f'\n⚠️  Solo hay {len(challenges)} retos — midiendo todos')
            n_medir = len(challenges)
        else:
            n_medir = N_CHALLENGES

        print(f'\nMidiendo {n_medir} retos SIN anillo isotrópico...')
        print(f'Puerto entrada: {INPORT}  |  ~{n_medir * 0.5:.0f}s estimados\n')

        # ── PREPARAR CHIP ────────────────────────────────────────────────
        sl.reset_mesh()
        sl.set_input_port(INPORT)
        sl.enable_internal_monitoring()

        # ── BUCLE DE MEDIDA ──────────────────────────────────────────────
        powers_list = []
        t0          = time.time()

        barra    = widgets.IntProgress(
            value=0, min=0, max=n_medir,
            description='Midiendo:', bar_style='info',
            layout=widgets.Layout(width='500px')
        )
        etiqueta = widgets.Label(value=f'0 / {n_medir}')
        display(widgets.HBox([barra, etiqueta]))

        for idx in range(n_medir):
            reto = challenges[idx]

            # ── Reset completo — sin programar ruta del anillo ───────────
            sl.reset_mesh()

            # ── Aplicar solo las fases dinámicas del reto ────────────────
            # Las TBUs de ruta quedan en reset (solo φ₀, sin corriente)
            # Esto es la diferencia clave respecto al experimento con anillo
            fases_driven = {int(k): v for k, v in reto.items()}
            sl.set_driven_phases(fases_driven, compensate_passive_phase=False)

            # ── Leer potencias ────────────────────────────────────────────
            powers = sl.get_output_power(inport=INPORT, outport=outports)
            powers_list.append({str(k): v for k, v in powers.items()})

            barra.value    = idx + 1
            elapsed        = time.time() - t0
            remaining      = (elapsed / (idx + 1)) * (n_medir - idx - 1)
            etiqueta.value = (
                f'{idx+1}/{n_medir}  |  '
                f'{elapsed:.0f}s  |  ~{remaining:.0f}s restantes'
            )

        barra.bar_style = 'success'
        t_total = time.time() - t0
        print(f'\n✓ {len(powers_list)} retos medidos en {t_total:.1f}s  '
              f'({t_total/n_medir:.2f}s por reto)')

        # ── GUARDAR ──────────────────────────────────────────────────────
        with open(fout, 'w') as f:
            json.dump({
                'inport':         INPORT,
                'outports':       outports,
                'n_retos':        len(powers_list),
                'modo':           'sin_anillo_isotropico',
                'seed':           d_chall['seed'],
                'dynamic_pucs':   dynamic_pucs,
                'route_pucs':     route_pucs,
                'elapsed_s':      round(t_total, 2),
                'powers':         powers_list,
            }, f, indent=2)

        print(f'✓ Guardado en {fout}')
        print(f'\n── Primeras 5 potencias del reto 0 ──────────────────')
        for p, v in list(powers_list[0].items())[:5]:
            print(f'  Puerto {p}: {v:.4f} dBm')

        # ── Ver estado de progreso ────────────────────────────────────────
        medidos_ahora = sorted([
            int(f.stem.split('sin_anillo_port')[1].split('.')[0])
            for f in RESULTS_DIR.glob('powers_sin_anillo_port*.json')
        ])
        pendientes_ahora = [p for p in ACTIVE_PORTS if p not in medidos_ahora]
        print(f'\nPuertos medidos:   {medidos_ahora}')
        print(f'Puertos pendientes:{pendientes_ahora}')
        if pendientes_ahora:
            print(f'\n→ Cambia INPORT = {pendientes_ahora[0]} en Celda 0 '
                  f'y ejecuta desde Celda 2')
        else:
            print(f'\n✓ ¡TODOS LOS PUERTOS MEDIDOS!')


In [ ]:
# ── VERIFICACIÓN VISUAL — Matriz de potencias (puerto actual) ─────────
import json, pathlib, numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = pathlib.Path('resultados_sin_anillo')

fpath = RESULTS_DIR / f'powers_sin_anillo_port{INPORT:02d}.json'
if not fpath.exists():
    print(f'⚠️  No existe {fpath.name} — ¿has ejecutado la Celda 4?')
else:
    with open(fpath) as f:
        d = json.load(f)

    outports = d['outports']
    powers   = d['powers']
    n_retos  = d['n_retos']

    # ── Construir matriz (n_puertos × n_retos) ────────────────────────
    mat = np.array([
        [p.get(str(port), -60.0) for port in outports]
        for p in powers
    ]).T   # transponer → filas=puertos, columnas=retos

    print(f'Puerto {INPORT}  |  {n_retos} retos  |  {len(outports)} puertos de salida')
    print(f'Potencias: min={mat.min():.1f} dBm  max={mat.max():.1f} dBm  '
          f'media={mat.mean():.1f} dBm')

    # ── Figura ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Panel A — Heatmap potencias (puerto × reto)
    ax = axes[0]
    im = ax.imshow(mat, aspect='auto', cmap='viridis',
                   interpolation='nearest',
                   vmin=np.percentile(mat, 5),
                   vmax=np.percentile(mat, 95))
    plt.colorbar(im, ax=ax, label='Potencia (dBm)')
    ax.set_xlabel('Índice de reto', fontsize=12)
    ax.set_ylabel('Puerto de salida', fontsize=12)
    ax.set_yticks(range(len(outports)))
    ax.set_yticklabels([str(p) for p in outports], fontsize=7)
    ax.set_title(f'A.  Heatmap potencias\n'
                 f'Puerto entrada {INPORT}  |  {n_retos} retos',
                 fontsize=12, fontstyle='italic')

    # Panel B — Potencia media por puerto de salida
    ax = axes[1]
    media  = mat.mean(axis=1)
    std    = mat.std(axis=1)
    x      = range(len(outports))
    bars   = ax.bar(x, media, color='#7EB8E8', alpha=0.75, edgecolor='none')
    # Colorear el más brillante y el más oscuro
    bars[np.argmax(media)].set_facecolor('#2AAA50')
    bars[np.argmin(media)].set_facecolor('#CC2222')
    ax.errorbar(x, media, yerr=std, fmt='none',
                color='#444444', linewidth=1.2, capsize=3)
    ax.axhline(media.mean(), color='#F0A875', linestyle='--',
               linewidth=1.5, label=f'Media global = {media.mean():.1f} dBm')
    ax.set_xticks(x)
    ax.set_xticklabels([str(p) for p in outports], rotation=45, fontsize=7)
    ax.set_xlabel('Puerto de salida', fontsize=12)
    ax.set_ylabel('Potencia media (dBm)', fontsize=12)
    ax.set_title('B.  Potencia media ± std por puerto\n'
                 '(verde=máx, rojo=mín)',
                 fontsize=12, fontstyle='italic')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis='y')

    # Panel C — Varianza por puerto (estabilidad entre retos)
    ax = axes[2]
    varianza = mat.var(axis=1)
    colores  = ['#CC2222' if v > np.percentile(varianza, 75) else
                '#F0A875' if v > np.percentile(varianza, 50) else
                '#7EB8E8'
                for v in varianza]
    ax.bar(x, varianza, color=colores, alpha=0.8, edgecolor='none')
    ax.axhline(varianza.mean(), color='#444444', linestyle='--',
               linewidth=1.5, label=f'Media = {varianza.mean():.4f} dB²')
    ax.set_xticks(x)
    ax.set_xticklabels([str(p) for p in outports], rotation=45, fontsize=7)
    ax.set_xlabel('Puerto de salida', fontsize=12)
    ax.set_ylabel('Varianza entre retos (dB²)', fontsize=12)
    ax.set_title('C.  Varianza inter-reto por puerto\n'
                 '(rojo=más variable, azul=más estable)',
                 fontsize=12, fontstyle='italic')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis='y')

    plt.suptitle(f'Verificación medida SIN anillo — Puerto {INPORT}\n'
                 f'{n_retos} retos  |  {len(outports)} puertos de salida  |  '
                 f'Modo: {d["modo"]}',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'verificacion_port{INPORT:02d}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Resumen numérico ──────────────────────────────────────────────
    print(f'\n{"─"*50}')
    print(f'  Puertos > -35 dBm (brillantes): '
          f'{(media > -35).sum()} de {len(outports)}')
    print(f'  Puertos < -40 dBm (oscuros):    '
          f'{(media < -40).sum()} de {len(outports)}')
    print(f'  Rango dinámico medio:            '
          f'{media.max() - media.min():.1f} dB')
    print(f'  Puerto más brillante: '
          f'{outports[np.argmax(media)]} ({media.max():.1f} dBm)')
    print(f'  Puerto más oscuro:    '
          f'{outports[np.argmin(media)]} ({media.min():.1f} dBm)')
    print(f'{"─"*50}')
    print(f'  ✓ Guardado en verificacion_port{INPORT:02d}.png')

## Celda 5 — Estado de progreso
Ejecutar en cualquier momento para ver cuántos puertos faltan.

In [ ]:
import pathlib
import json
import numpy as np

ACTIVE_PORTS = list(range(0, 22)) + list(range(34, 40))
RESULTS_DIR  = pathlib.Path('resultados_sin_anillo')

archivos = sorted(RESULTS_DIR.glob('powers_sin_anillo_port*.json'))
medidos  = sorted([
    int(f.stem.split('sin_anillo_port')[1].split('.')[0])
    for f in archivos
])
pendientes = [p for p in ACTIVE_PORTS if p not in medidos]

print(f'{"═"*55}')
print(f'  ESTADO DE PROGRESO — Sin anillo isotrópico')
print(f'{"═"*55}')
print(f'  Puertos medidos:    {len(medidos)}/{len(ACTIVE_PORTS)}')
print(f'  Puertos pendientes: {len(pendientes)}')
print(f'{"─"*55}')

if archivos:
    print(f'\n{"Puerto":>8}  {"Retos":>7}  {"Tiempo (s)":>12}  {"Fichero"}')
    print('-' * 55)
    for f in archivos:
        with open(f) as fp:
            d = json.load(fp)
        p = int(f.stem.split('sin_anillo_port')[1].split('.')[0])
        print(f'{p:>8}  {d["n_retos"]:>7}  {d["elapsed_s"]:>12.1f}  {f.name}')

print(f'\n{"─"*55}')
if pendientes:
    print(f'  Siguiente puerto: {pendientes[0]}')
    print(f'  → Cambia INPORT = {pendientes[0]} en Celda 0')
    print(f'    y ejecuta desde Celda 2')
else:
    print(f'  ✓ ¡TODOS LOS PUERTOS MEDIDOS!')
    total_retos = sum(
        json.load(open(f))['n_retos'] for f in archivos
    )
    print(f'  Total retos medidos: {total_retos:,}')
print(f'{"═"*55}')


## Celda 6 — Desconectar el chip
Ejecutar **siempre al terminar**, aunque haya habido errores.

In [ ]:
sl.reset_mesh()
sl.disconnect()
print('✓ Chip desconectado correctamente.')


## Celda extra — Verificación rápida de un fichero
Comprueba que los datos guardados son coherentes.

In [ ]:
import json, pathlib, numpy as np

RESULTS_DIR  = pathlib.Path('resultados_sin_anillo')
INPORT_CHECK = 3  # ← cambia para ver otro puerto

fpath = RESULTS_DIR / f'powers_sin_anillo_port{INPORT_CHECK:02d}.json'
if not fpath.exists():
    print(f'⚠️  No existe {fpath.name}')
else:
    with open(fpath) as f:
        d = json.load(f)

    outports = d['outports']
    powers   = d['powers']
    n_retos  = d['n_retos']

    # Matriz de potencias (n_retos × n_puertos)
    mat = np.array([
        [p.get(str(port), -60.0) for port in outports]
        for p in powers
    ])

    print(f'{"═"*55}')
    print(f'  VERIFICACIÓN — Puerto {INPORT_CHECK}')
    print(f'{"═"*55}')
    print(f'  Retos:          {n_retos}')
    print(f'  Puertos salida: {len(outports)}')
    print(f'  Modo:           {d["modo"]}')
    print(f'  Seed:           {d["seed"]}')
    print(f'{"─"*55}')
    print(f'  Potencias (dBm):')
    print(f'    Min global:  {mat.min():.2f} dBm')
    print(f'    Max global:  {mat.max():.2f} dBm')
    print(f'    Media:       {mat.mean():.2f} dBm')
    print(f'    Std media:   {mat.std(axis=0).mean():.4f} dBm')
    print(f'{"─"*55}')

    # Distribución de potencia del primer reto
    p0 = mat[0]
    print(f'  Reto 0 — distribución de potencia:')
    print(f'    Puerto más brillante: {outports[np.argmax(p0)]} ({p0.max():.2f} dBm)')
    print(f'    Puerto más oscuro:    {outports[np.argmin(p0)]} ({p0.min():.2f} dBm)')
    print(f'    Rango dinámico:       {p0.max()-p0.min():.2f} dB')
    print(f'    Puertos > -35 dBm:    {(p0 > -35).sum()} de {len(outports)}')
    print(f'{"═"*55}')
